In [1]:
import pandas as pd
import numpy as np
import re
import os
from scipy.interpolate import interp1d

# =========================
# Load Data
# =========================

df = pd.read_csv("../data/raw/dataset.csv")
original = df.copy()

ce_cols = [c for c in df.columns if c.endswith("CE")]
pe_cols = [c for c in df.columns if c.endswith("PE")]

def get_strike(col):
    return int(re.search(r"JAN26(\d+)(CE|PE)$", col).group(1))

ce_cols = sorted(ce_cols, key=get_strike)
pe_cols = sorted(pe_cols, key=get_strike)

ce_strikes = np.array([get_strike(c) for c in ce_cols])
pe_strikes = np.array([get_strike(c) for c in pe_cols])

# =========================
# Linear Interpolation
# =========================

filled = df.copy()

def interpolate_row(row, cols, strikes):

    values = row[cols].astype(float).values.copy()

    mask = ~np.isnan(values)

    if mask.sum() == 0:
        return values

    if mask.sum() == 1:
        values[~mask] = values[mask][0]
        return values

    f = interp1d(
        strikes[mask],
        values[mask],
        kind="linear",
        fill_value="extrapolate",
        bounds_error=False
    )

    values[~mask] = f(strikes[~mask])

    return values

for idx in filled.index:

    filled.loc[idx, ce_cols] = interpolate_row(
        filled.loc[idx],
        ce_cols,
        ce_strikes
    )

    filled.loc[idx, pe_cols] = interpolate_row(
        filled.loc[idx],
        pe_cols,
        pe_strikes
    )

# Time-based fallback
for col in ce_cols + pe_cols:
    filled[col] = filled[col].interpolate(
        method="linear",
        limit_direction="both"
    )

# IV cannot be negative
filled[ce_cols + pe_cols] = (
    filled[ce_cols + pe_cols]
    .clip(lower=0.001)
)

print(
    "Remaining missing:",
    filled[ce_cols + pe_cols]
    .isna()
    .sum()
    .sum()
)

# =========================
# Submission Creation
# =========================

rows = []

for idx in original.index:

    dt = original.loc[idx, "datetime"]

    for col in ce_cols + pe_cols:

        if pd.isna(original.loc[idx, col]):

            rows.append({
                "id": f"{dt}||{col}",
                "value": filled.loc[idx, col]
            })

submission = pd.DataFrame(rows)

submission.to_csv(
    "submission_best_linear_fit.csv",
    index=False
)

print("\nSubmission Shape:", submission.shape)
print(submission.head())
print("\nSaved: submission_best_linear_fit.csv")

Remaining missing: 0

Submission Shape: (5460, 2)
                                      id    value
0  07-01-2026 09:15||NIFTY27JAN2625500CE  0.11373
1  07-01-2026 09:15||NIFTY27JAN2625800CE  0.10150
2  07-01-2026 09:15||NIFTY27JAN2624100PE  0.16344
3  07-01-2026 09:20||NIFTY27JAN2625300CE  0.09681
4  07-01-2026 09:20||NIFTY27JAN2625400CE  0.10730

Saved: submission_best_linear_fit.csv
